In [1]:
import ee
import geemap
import pandas as pd
import numpy as np
from tqdm import tqdm

In [2]:
ee.Authenticate()

ee.Initialize(
    project='soil-health-project-497508'
)

print(
    "Earth Engine Connected"
)

Earth Engine Connected


In [3]:
df = pd.read_csv(
    "cleaned_MH_OC.csv"
)

print(
    df.shape
)

df.head()

(29942, 161)


,lat_key,fid,latitude,longitude,CLIMATE_VALUE,CLIMATE_SUBCLASS,CLIMATE_CLASS,DOMSOI,SOIL_TYPE,SOIL_SUBCLASS,...,S1_VH_ASM,S1_VH_Energy,S1_VH_MaxProbability,S1_VH_Entropy,S1_VH_GLCM_Mean,S1_VH_GLCM_Variance,S1_VH_GLCM_Correlation,year,month,dayofyear
0,15.709677,48980.0,15.709677,74.047631,2.0,Am,Tropical,Ap,ACRISOLS,Plinthic Acrisols,...,0.699068,0.999988,0.995973,-4.731746,0.022191,0.000029,0.424396,2025,2,48
1,15.709731,48981.0,15.709731,74.048897,2.0,Am,Tropical,Ap,ACRISOLS,Plinthic Acrisols,...,0.636399,0.999990,0.997320,-4.856527,0.016932,0.000098,0.181971,2025,2,48
2,15.711065,48982.0,15.711065,74.047657,2.0,Am,Tropical,Ap,ACRISOLS,Plinthic Acrisols,...,0.606309,0.999960,0.995374,-4.144549,0.013338,0.000111,0.318890,2025,2,48
3,15.804995,48839.5,15.804995,73.733783,2.0,Am,Tropical,Nd,NITOSOLS,Dystric Nitosols,...,0.774612,0.999999,0.998807,-6.255501,0.007464,0.000004,0.314625,2024,3,91
4,15.805012,48835.0,15.805012,73.734112,2.0,Am,Tropical,Nd,NITOSOLS,Dystric Nitosols,...,0.648962,0.999997,0.998350,-5.404119,0.007534,0.000008,1.216725,2024,3,91


In [4]:
embed_input = df[
    ['fid','latitude','longitude','year']
].copy()

embed_input.head()

,fid,latitude,longitude,year
0,48980.0,15.709677,74.047631,2025
1,48981.0,15.709731,74.048897,2025
2,48982.0,15.711065,74.047657,2025
3,48839.5,15.804995,73.733783,2024
4,48835.0,15.805012,73.734112,2024


In [5]:
embeddings = ee.ImageCollection(
    "GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL"
)

print(
    embeddings.size().getInfo()
)

97155


In [9]:
batch_size = 1000

all_results = []

years = sorted(
    embed_input.year.unique()
)

for yr in years:

    print(
        f"\nProcessing year {yr}"
    )

    yearly_df = embed_input[
        embed_input.year == yr
    ]

    img = embeddings.filterDate(

        f"{yr}-01-01",

        f"{yr}-12-31"

    ).mosaic()

    for start in tqdm(

        range(
            0,
            len(yearly_df),
            batch_size
        )
    ):

        end = start + batch_size

        batch = yearly_df.iloc[
            start:end
        ]

        features = []

        for _, row in batch.iterrows():

            feat = ee.Feature(

                ee.Geometry.Point([
                    row.longitude,
                    row.latitude
                ]),

                {"fid": row.fid}
            )

            features.append(
                feat
            )

        fc = ee.FeatureCollection(
            features
        )

        sampled = img.sampleRegions(

            collection=fc,

            scale=100,

            geometries=False
        )

        features_out = sampled.getInfo()[
            'features'
        ]

        rows = [

            f['properties']

            for f in features_out
        ]

        temp = pd.DataFrame(
            rows
        )

        all_results.append(
            temp
        )


Processing year 2023


100%|██████████| 1/1 [00:00<00:00,  2.01it/s]



Processing year 2024


100%|██████████| 14/14 [01:37<00:00,  7.00s/it]



Processing year 2025


100%|██████████| 17/17 [01:52<00:00,  6.61s/it]


In [10]:
embed_df = pd.concat(
    all_results,
    ignore_index=True
)

print(
    embed_df.shape
)

embed_df.head()

(29942, 65)


,A00,A01,A02,A03,A04,A05,A06,A07,A08,A09,...,A55,A56,A57,A58,A59,A60,A61,A62,A63,fid
0,-0.000554,0.019931,0.259900,0.019931,-0.147697,0.022207,-0.038447,0.051734,-0.059116,-0.012057,...,-0.003014,0.066990,0.019931,-0.228897,-0.199862,-0.022207,-0.059116,0.093564,0.051734,14277.0
1,-0.006151,-0.000984,0.172795,-0.027128,-0.135886,-0.024606,-0.079723,0.172795,-0.084214,0.004983,...,-0.130165,-0.001538,-0.130165,-0.166336,-0.141730,0.004983,-0.098424,0.103406,0.088827,52255.0
2,-0.059116,0.006151,0.192910,-0.027128,-0.147697,0.044844,-0.062991,0.103406,-0.019931,0.017778,...,-0.103406,0.044844,-0.103406,-0.221453,-0.199862,-0.003014,-0.071111,0.098424,0.066990,52248.0
3,0.048228,-0.022207,0.292872,0.075356,-0.066990,-0.093564,-0.013841,0.084214,-0.113741,-0.103406,...,-0.130165,0.062991,-0.093564,-0.267958,-0.147697,-0.007443,-0.130165,-0.079723,0.098424,29740.0
4,0.048228,-0.022207,0.292872,0.075356,-0.066990,-0.093564,-0.013841,0.084214,-0.113741,-0.103406,...,-0.130165,0.062991,-0.093564,-0.267958,-0.147697,-0.007443,-0.130165,-0.079723,0.098424,29684.0


In [11]:
embed_df.to_csv(

    "alphaearth_embeddings.csv",

    index=False
)

print(
    "Saved successfully"
)

Saved successfully
